In [1]:
DATA = "bookings.jsonl"

# Task 2: Contracts, nested data and a star schema

Every cell below reads from `DATA`; no count is typed in by hand.

In [2]:
import json
import math
import re
import shutil
from collections import Counter
from datetime import date, datetime, timezone
from pathlib import Path
from typing import List, Literal, Optional

import duckdb
import pandas as pd
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator, model_validator

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Part A: The contract

Four Pydantic models: `Ticket`, `Student`, `Event`, and `Booking`, which uses the other three.
Single-field rules are field types / constraints; the two cross-field rules
(`total` = Σ qty × price, and `booked_at` before the event date) live in a `model_validator`.

In [3]:
STUDENT_ID = r"^[0-9]{2}-[0-9]{4}$"   # two digits, a dash, four digits


class Ticket(BaseModel):
    model_config = ConfigDict(extra="forbid")
    type: Literal["standard", "vip", "student"]
    qty: int = Field(ge=1, strict=True)          # at least 1, a real integer
    price: float = Field(ge=0, strict=True)      # not negative (ints are accepted as floats)


class Student(BaseModel):
    id: str = Field(pattern=STUDENT_ID)
    name: str = Field(min_length=1)
    faculty: Literal["MET", "IET", "EMS", "Pharmacy", "Management"]


class Event(BaseModel):
    # all five fields are required (no defaults) and must not be empty
    id: str = Field(min_length=1)
    title: str = Field(min_length=1)
    club: Literal["Robotics", "Debate", "Music", "Football", "Chess"]
    date: date
    venue: str = Field(min_length=1)


class Booking(BaseModel):
    # extra="allow": a field that drifts in later is kept, not silently dropped
    model_config = ConfigDict(extra="allow")

    booking_id: str = Field(min_length=1)
    booked_at: datetime
    student: Student
    event: Event                                  # required and not None -> "present"
    tickets: List[Ticket] = Field(min_length=1)   # at least one ticket
    total: float = Field(strict=True)
    payment: Literal["card", "cash", "wallet"]
    status: Literal["confirmed", "cancelled"]
    promo_code: Optional[str] = None              # the drifting field (see Part B.3)

    @field_validator("booked_at")
    @classmethod
    def as_utc(cls, v: datetime) -> datetime:
        # booked_at is in UTC; a timestamp without an offset is read as UTC
        return v.replace(tzinfo=timezone.utc) if v.tzinfo is None else v.astimezone(timezone.utc)

    @model_validator(mode="after")
    def cross_field_rules(self):
        expected = sum(t.qty * t.price for t in self.tickets)
        if not math.isclose(self.total, expected, abs_tol=0.005):
            raise ValueError(f"total {self.total} != sum of qty x price {expected}")
        if not self.booked_at.date() < self.event.date:
            raise ValueError(f"booked_at {self.booked_at.isoformat()} is not before event date {self.event.date}")
        return self

### Validate every line

Each failure goes to the quarantine with its `booking_id` and a reason. The reason is the
field path plus the error message (cross-field errors have no field path, so they are labelled
`booking`). Lines that are not JSON objects are quarantined as `invalid JSON`.

In [4]:
def reason_from(err: ValidationError) -> str:
    parts = []
    for e in err.errors():
        # drop list indexes (tickets.0.qty -> tickets.qty) so the same problem groups together
        loc = ".".join(str(p) for p in e["loc"] if not isinstance(p, int)) or "booking"
        msg = e["msg"].removeprefix("Value error, ")
        parts.append(f"{loc}: {msg}")
    return "; ".join(parts)


def short_reason(full: str) -> str:
    """A stable category for the per-reason report (numbers/values stripped from cross-field messages)."""
    cats = []
    for part in full.split("; "):
        if part.startswith("booking: total"):
            part = "booking: total != sum of qty x price"
        elif part.startswith("booking: booked_at"):
            part = "booking: booked_at not before event date"
        cats.append(part)
    return "; ".join(cats)


lines_read = 0
valid = []        # (line_no, raw_dict, Booking)
quarantine = []   # dicts: line, booking_id, reason

with open(DATA, encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue                      # blank lines are not bookings
        lines_read += 1
        try:
            raw = json.loads(line)
            if not isinstance(raw, dict):
                raise ValueError("not a JSON object")
        except ValueError:
            quarantine.append({"line": line_no, "booking_id": None, "reason": "invalid JSON"})
            continue
        try:
            valid.append((line_no, raw, Booking.model_validate(raw)))
        except ValidationError as err:
            quarantine.append({"line": line_no, "booking_id": raw.get("booking_id"),
                               "reason": short_reason(reason_from(err)),
                               "detail": reason_from(err)})

print("lines read:", lines_read)
print("passed the contract:", len(valid))
print("failed the contract:", len(quarantine))

lines read: 283
passed the contract: 262
failed the contract: 21


### Re-sent bookings

The duplicate check runs **after** validation, on valid bookings only: keep the first valid copy of
each `booking_id`, quarantine later copies with the reason `duplicate booking_id`.

In [5]:
seen = set()
kept = []
for line_no, raw, b in valid:
    if b.booking_id in seen:
        quarantine.append({"line": line_no, "booking_id": b.booking_id, "reason": "duplicate booking_id"})
    else:
        seen.add(b.booking_id)
        kept.append((line_no, raw, b))

quarantine.sort(key=lambda q: q["line"])
with open("quarantine.jsonl", "w", encoding="utf-8") as f:
    for q in quarantine:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")

print("saved", len(quarantine), "quarantined lines to quarantine.jsonl")

saved 24 quarantined lines to quarantine.jsonl


### What if the duplicate check ran first?

For comparison only (the pipeline above is not changed): de-duplicate the raw lines by `booking_id`
first, then validate. If a first copy is invalid and a later copy is valid, this order throws the
valid copy away as a "duplicate" and loses a good booking.

In [6]:
swap_seen, swap_dupes, swap_valid, swap_invalid = set(), 0, 0, 0
with open(DATA, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        try:
            raw = json.loads(line)
            if not isinstance(raw, dict):
                raise ValueError
        except ValueError:
            swap_invalid += 1
            continue
        bid = raw.get("booking_id")
        if bid in swap_seen:
            swap_dupes += 1
            continue
        swap_seen.add(bid)
        try:
            Booking.model_validate(raw)
            swap_valid += 1
        except ValidationError:
            swap_invalid += 1

n_dupes = sum(1 for q in quarantine if q["reason"] == "duplicate booking_id")
print(f"validate -> dedupe (used): kept {len(kept)}, duplicates {n_dupes}, failed contract {len(quarantine) - n_dupes}")
print(f"dedupe -> validate (swap): kept {swap_valid}, duplicates {swap_dupes}, failed contract {swap_invalid}")
print("counts change when swapped:", (swap_valid, swap_dupes) != (len(kept), n_dupes))

validate -> dedupe (used): kept 259, duplicates 3, failed contract 21
dedupe -> validate (swap): kept 259, duplicates 3, failed contract 21
counts change when swapped: False


### Report

In [7]:
reason_counts = Counter(q["reason"] for q in quarantine)
report = pd.DataFrame(sorted(reason_counts.items(), key=lambda kv: (-kv[1], kv[0])), columns=["reason", "count"])

print(f"lines read:       {lines_read}")
print(f"passed:           {len(kept)}")
print(f"quarantined:      {len(quarantine)}")
assert lines_read == len(kept) + len(quarantine)
display(report)
display(pd.DataFrame(quarantine)[["line", "booking_id", "reason"]])

lines read:       283
passed:           259
quarantined:      24


,reason,count
0,booking: total != sum of qty x price,4
1,student.id: String should match pattern '^[0-9...,4
2,booking: booked_at not before event date,3
3,duplicate booking_id,3
4,tickets.qty: Input should be greater than or e...,3
5,"tickets.type: Input should be 'standard', 'vip...",3
6,event: Input should be a valid dictionary or i...,2
7,tickets: List should have at least 1 item afte...,2


,line,booking_id,reason
0,11,B0011,"tickets.type: Input should be 'standard', 'vip..."
1,16,B0016,tickets.qty: Input should be greater than or e...
2,24,B0024,student.id: String should match pattern '^[0-9...
3,44,B0035,duplicate booking_id
4,50,B0049,event: Input should be a valid dictionary or i...
5,58,B0057,student.id: String should match pattern '^[0-9...
6,61,B0060,student.id: String should match pattern '^[0-9...
7,91,B0090,"tickets.type: Input should be 'standard', 'vip..."
8,102,B0101,booking: total != sum of qty x price
9,107,B0106,student.id: String should match pattern '^[0-9...


## Part B: Flatten

`bookings`: one row per booking, `student` and `event` objects become prefixed columns, no `tickets`.
`booking_tickets`: one row per ticket entry, with `amount = qty × price`.

In [8]:
records = [b.model_dump() for _, _, b in kept]

bookings = pd.json_normalize(records, sep="_").drop(columns=["tickets"])
bookings["booked_at"] = pd.to_datetime(bookings["booked_at"], utc=True)
bookings["event_date"] = pd.to_datetime(bookings["event_date"])
front = ["booking_id", "booked_at", "student_id", "student_name", "student_faculty",
         "event_id", "event_title", "event_club", "event_date", "event_venue", "total", "payment", "status"]
bookings = bookings[front + [c for c in bookings.columns if c not in front]]

booking_tickets = pd.DataFrame(
    [{"booking_id": b.booking_id, "type": t.type, "qty": t.qty, "price": t.price, "amount": t.qty * t.price}
     for _, _, b in kept for t in b.tickets],
    columns=["booking_id", "type", "qty", "price", "amount"],
)

print("bookings:", bookings.shape, " booking_tickets:", booking_tickets.shape)
display(bookings.head())
display(booking_tickets.head())

bookings: (259, 14)  booking_tickets: (314, 5)


,booking_id,booked_at,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue,total,payment,status,promo_code
0,B0001,2026-09-28 15:46:00+00:00,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,340.0,card,cancelled,NaN
1,B0002,2026-09-27 12:29:00+00:00,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,40.0,card,confirmed,NaN
2,B0003,2026-10-13 20:19:00+00:00,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium,60.0,wallet,confirmed,NaN
3,B0004,2026-11-01 06:42:00+00:00,58-6052,Seif Ibrahim,Management,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage,105.0,card,confirmed,CLUB10
4,B0005,2026-10-28 19:06:00+00:00,58-9891,Ali Fouad,Management,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A,20.0,cash,confirmed,NaN


,booking_id,type,qty,price,amount
0,B0001,standard,1,40.0,40.0
1,B0001,vip,3,100.0,300.0
2,B0002,standard,1,40.0,40.0
3,B0003,standard,2,30.0,60.0
4,B0004,standard,3,35.0,105.0


### Checks

In [9]:
assert bookings["booking_id"].is_unique, "booking_id is not unique in bookings"
assert "tickets" not in bookings.columns

amount_per_booking = booking_tickets.groupby("booking_id")["amount"].sum()
check = bookings.set_index("booking_id")["total"].to_frame().join(amount_per_booking)
assert check["amount"].notna().all(), "a booking has no tickets"
assert ((check["total"] - check["amount"]).abs() < 0.005).all(), "ticket amounts do not add up to total"
print("OK: booking_id unique;", len(check), "bookings whose ticket amounts add up to their total")

OK: booking_id unique; 259 bookings whose ticket amounts add up to their total


### Schema drift

Compare the keys of every valid raw document (top level and inside `student`, `event`, `tickets`)
with the keys of the example in the brief. For each extra key: how many documents have it, and the
earliest `booked_at` it appears at.

In [10]:
EXPECTED = {
    "": {"booking_id", "booked_at", "student", "event", "tickets", "total", "payment", "status"},
    "student": {"id", "name", "faculty"},
    "event": {"id", "title", "club", "date", "venue"},
    "tickets": {"type", "qty", "price"},
}

def keys_of(raw):
    out = {("", k) for k in raw}
    for part in ("student", "event"):
        if isinstance(raw.get(part), dict):
            out |= {(part, k) for k in raw[part]}
    for t in raw.get("tickets") or []:
        if isinstance(t, dict):
            out |= {("tickets", k) for k in t}
    return out

rows = []
for line_no, raw, b in kept:
    for parent, k in keys_of(raw):
        if k not in EXPECTED[parent]:
            rows.append({"field": f"{parent}.{k}".lstrip("."), "line": line_no,
                         "booking_id": b.booking_id, "booked_at": b.booked_at})
extra = pd.DataFrame(rows, columns=["field", "line", "booking_id", "booked_at"])

drift = []
by_time = bookings.sort_values("booked_at")[["booking_id", "booked_at"]]
for field, g in extra.groupby("field"):
    first = g["booked_at"].min()
    after = by_time[by_time["booked_at"] >= first]
    drift.append({
        "field": field,
        "documents_with_it": len(g),
        "documents_without_it": len(bookings) - len(g),
        "first_booked_at": first,
        "first_booking_id": g.loc[g["booked_at"].idxmin(), "booking_id"],
        "last_booked_at_without_it": by_time.loc[~by_time["booking_id"].isin(g["booking_id"]), "booked_at"].max(),
        "every_booking_from_then_has_it": after["booking_id"].isin(g["booking_id"]).all(),
    })
drift = pd.DataFrame(drift)
display(drift)

for d in drift.itertuples():
    print(f"'{d.field}' first appears in booking {d.first_booking_id}, booked at {d.first_booked_at} "
          f"(from {d.first_booked_at.date()} on); {d.documents_with_it} valid bookings have it, "
          f"{d.documents_without_it} do not.")
    col = d.field.replace(".", "_")
    if col in bookings.columns:
        print(f"  values in bookings.{col} (NaN/None = field missing or null):")
        print(bookings[col].value_counts(dropna=False).to_string())

,field,documents_with_it,documents_without_it,first_booked_at,first_booking_id,last_booked_at_without_it,every_booking_from_then_has_it
0,promo_code,173,86,2026-10-20 01:05:00+00:00,B0125,2026-10-19 18:43:00+00:00,True


'promo_code' first appears in booking B0125, booked at 2026-10-20 01:05:00+00:00 (from 2026-10-20 on); 173 valid bookings have it, 86 do not.
  values in bookings.promo_code (NaN/None = field missing or null):
promo_code
NaN       172
CLUB10     47
EARLY      40


Documents without the drifted field are **kept**: the field is optional in the contract
(`promo_code: Optional[str] = None`), so older documents get a null in that column instead of being rejected.

## Part C: Model and store (DuckDB star schema)

Grain of the fact: one row per ticket entry. `dim_event` and `dim_student` get surrogate keys from
`row_number()`. The date key is `booked_at` (UTC) as `YYYYMMDD`.

In [11]:
con = duckdb.connect()
con.execute("SET TimeZone = 'UTC'")

# DuckDB gets naive UTC timestamps so date keys and months are computed in UTC
bookings_db = bookings.assign(booked_at=bookings["booked_at"].dt.tz_convert("UTC").dt.tz_localize(None))
con.register("bookings", bookings_db)
con.register("booking_tickets", booking_tickets)

con.execute("""
CREATE OR REPLACE TABLE dim_event AS
SELECT row_number() OVER (ORDER BY event_id) AS event_key, *
FROM (
    SELECT event_id,
           arg_max(event_title, booked_at) AS event_title,
           arg_max(event_club,  booked_at) AS event_club,
           arg_max(event_date,  booked_at)::DATE AS event_date,
           arg_max(event_venue, booked_at) AS event_venue
    FROM bookings
    GROUP BY event_id
)
""")

con.execute("""
CREATE OR REPLACE TABLE dim_student AS
SELECT row_number() OVER (ORDER BY student_id) AS student_key, *
FROM (
    SELECT student_id,
           arg_max(student_name,    booked_at) AS student_name,
           arg_max(student_faculty, booked_at) AS student_faculty
    FROM bookings
    GROUP BY student_id
)
""")

con.execute("""
CREATE OR REPLACE TABLE fact_ticket_sales AS
SELECT t.booking_id,
       e.event_key,
       s.student_key,
       t.type   AS ticket_type,
       t.qty,
       t.price,
       t.amount,
       b.status,
       CAST(strftime(b.booked_at, '%Y%m%d') AS INTEGER) AS booked_date_key,
       strftime(b.booked_at, '%Y-%m')                   AS booked_month
FROM booking_tickets t
JOIN bookings    b ON b.booking_id = t.booking_id
JOIN dim_event   e ON e.event_id   = b.event_id
JOIN dim_student s ON s.student_id = b.student_id
""")

for t in ["dim_event", "dim_student", "fact_ticket_sales"]:
    print(t, con.execute(f"SELECT count(*) FROM {t}").fetchall()[0][0], "rows")
display(con.sql("SELECT * FROM dim_event").df())
display(con.sql("SELECT * FROM dim_student LIMIT 5").df())
display(con.sql("SELECT * FROM fact_ticket_sales LIMIT 5").df())

dim_event

 8 rows
dim_student 103 rows
fact_ticket_sales 314 rows


,event_key,event_id,event_title,event_club,event_date,event_venue
0,1,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A
1,2,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium
2,3,EV-03,Autumn Concert,Music,2026-10-29,Open-air theatre
3,4,EV-04,Five-a-side Cup,Football,2026-11-05,Sports field
4,5,EV-05,Blitz Chess Open,Chess,2026-11-07,C7.201
5,6,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A
6,7,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage
7,8,EV-08,Model UN Debate,Debate,2026-11-26,Auditorium


,student_key,student_id,student_name,student_faculty
0,1,58-1324,Nour Fouad,MET
1,2,58-1449,Farida Mansour,EMS
2,3,58-1801,Karim Kamel,IET
3,4,58-2165,Farida Hassan,Management
4,5,58-2307,Farida Hassan,Pharmacy


,booking_id,event_key,student_key,ticket_type,qty,price,amount,status,booked_date_key,booked_month
0,B0001,1,61,standard,1,40.0,40.0,cancelled,20260928,2026-09
1,B0001,1,61,vip,3,100.0,300.0,cancelled,20260928,2026-09
2,B0002,1,84,standard,1,40.0,40.0,confirmed,20260927,2026-09
3,B0003,2,14,standard,2,30.0,60.0,confirmed,20261013,2026-10
4,B0004,7,15,standard,3,35.0,105.0,confirmed,20261101,2026-11


### Checks

In [12]:
n_fact = con.execute("SELECT count(*) FROM fact_ticket_sales").fetchall()[0][0]
assert n_fact == len(booking_tickets), (n_fact, len(booking_tickets))

fact_amount = con.execute("SELECT sum(amount) FROM fact_ticket_sales").fetchall()[0][0]
bookings_total = con.execute("SELECT sum(total) FROM bookings").fetchall()[0][0]
assert math.isclose(fact_amount, bookings_total, abs_tol=0.01), (fact_amount, bookings_total)

# one row per natural key in each dimension, and every fact row found its dimensions
assert con.execute("SELECT count(*) = count(DISTINCT event_id) FROM dim_event").fetchall()[0][0]
assert con.execute("SELECT count(*) = count(DISTINCT student_id) FROM dim_student").fetchall()[0][0]
assert con.execute("SELECT count(*) FROM fact_ticket_sales WHERE event_key IS NULL OR student_key IS NULL").fetchall()[0][0] == 0

print(f"OK: fact rows = booking_tickets rows = {n_fact}; sum(amount) = sum(total) = {fact_amount:.2f}")

OK: fact rows = booking_tickets rows = 314; sum(amount) = sum(total) = 31285.00


### Tickets sold and revenue per club and ticket type (confirmed bookings only)

In [13]:
answer = con.sql("""
SELECT e.event_club     AS club,
       f.ticket_type,
       CAST(sum(f.qty) AS INTEGER) AS tickets_sold,
       sum(f.amount)    AS revenue_egp
FROM fact_ticket_sales f
JOIN dim_event e USING (event_key)
WHERE f.status = 'confirmed'
GROUP BY ALL
ORDER BY club, ticket_type
""").df()
display(answer)

,club,ticket_type,tickets_sold,revenue_egp
0,Chess,standard,45,900.0
1,Chess,student,44,440.0
2,Debate,standard,65,1950.0
3,Debate,student,34,510.0
4,Debate,vip,40,3000.0
5,Football,standard,64,1600.0
6,Music,standard,67,2930.0
7,Music,student,20,1000.0
8,Music,vip,78,9990.0
9,Robotics,standard,60,2400.0


### Save as Parquet

`star/dim_event.parquet`, `star/dim_student.parquet`, and `star/fact_ticket_sales/booked_month=…/`.

In [14]:
STAR = Path("star")
if STAR.exists():
    shutil.rmtree(STAR)          # a re-run starts from an empty folder
STAR.mkdir()

con.execute("COPY dim_event   TO 'star/dim_event.parquet'   (FORMAT PARQUET)")
con.execute("COPY dim_student TO 'star/dim_student.parquet' (FORMAT PARQUET)")
con.execute("COPY fact_ticket_sales TO 'star/fact_ticket_sales' (FORMAT PARQUET, PARTITION_BY (booked_month))")

for p in sorted(STAR.rglob("*")):
    print(p.as_posix())

# read the files back and check they match
back = con.execute("SELECT count(*), sum(amount) FROM read_parquet('star/fact_ticket_sales/*/*.parquet', hive_partitioning = true)").fetchall()[0]
assert back[0] == n_fact and math.isclose(back[1], fact_amount, abs_tol=0.01)
print("Parquet read back OK:", back)

star/dim_event.parquet
star/dim_student.parquet
star/fact_ticket_sales
star/fact_ticket_sales/booked_month=2026-09
star/fact_ticket_sales/booked_month=2026-09/data_0.parquet
star/fact_ticket_sales/booked_month=2026-10
star/fact_ticket_sales/booked_month=2026-10/data_0.parquet
star/fact_ticket_sales/booked_month=2026-11
star/fact_ticket_sales/booked_month=2026-11/data_0.parquet
Parquet read back OK: (314, 31285.0)


## Numbers for the README paragraph

In [15]:
print(f"lines read: {lines_read}")
for r in report.itertuples():
    print(f"  quarantined - {r.reason}: {r.count}")
print(f"quarantined total: {len(quarantine)}")
print(f"bookings left: {len(bookings)}  (ticket entries: {len(booking_tickets)})")
sum_total_over_fact = con.execute("""
    SELECT sum(b.total) FROM fact_ticket_sales f JOIN bookings b USING (booking_id)
""").fetchall()[0][0]
print(f"SUM(amount) over fact = {fact_amount:.2f}; SUM(total) if total were repeated on every fact row = {sum_total_over_fact:.2f}")

lines read: 283
  quarantined - booking: total != sum of qty x price: 4
  quarantined - student.id: String should match pattern '^[0-9]{2}-[0-9]{4}$': 4
  quarantined - booking: booked_at not before event date: 3
  quarantined - duplicate booking_id: 3
  quarantined - tickets.qty: Input should be greater than or equal to 1: 3
  quarantined - tickets.type: Input should be 'standard', 'vip' or 'student': 3
  quarantined - event: Input should be a valid dictionary or instance of Event: 2
  quarantined - tickets: List should have at least 1 item after validation, not 0: 2
quarantined total: 24
bookings left: 259  (ticket entries: 314)


SUM(amount) over fact = 31285.00; SUM(total) if total were repeated on every fact row = 42000.00
